# 00 · Reproducir el proyecto completo

**Evaluación Final Transversal — MLY1101 · Caso C: Spotify Tracks**

Este notebook ejecuta **todas las etapas** del proyecto desde el CSV original, sin pasos manuales,
usando los pipelines de Kedro. Es equivalente a ejecutar `kedro run` en la terminal.

| Etapa CRISP-DM | Pipeline Kedro | Qué genera |
|---|---|---|
| Preparación de datos | `data_cleaning` → `data_transformation` | `data/02_intermediate` … `data/05_model_input`, escalador |
| Modelado | `modeling` → `hyperparameter_tuning` | Modelos iniciales y optimizados (`models/trained_models/`) |
| Modelado (ensambles) | `ensembles` | Voting y Stacking |
| Evaluación | `evaluation` | Métricas, validación cruzada, generalización (`results/`) |
| Modelado no supervisado | `unsupervised` | K-Means, PCA, DBSCAN, perfiles de clusters |

**Tiempo aproximado:** 40–50 minutos en un notebook de 8 núcleos (la mayor parte es la optimización de hiperparámetros
y la validación cruzada de los ensambles).

Después de ejecutarlo, los notebooks `01` a `06` muestran y explican cada etapa:

| Notebook | Contenido |
|---|---|
| `01_eda_analisis_exploratorio` | Calidad de datos, distribuciones, correlaciones, sesgos |
| `02_limpieza_transformacion` | Limpieza, feature engineering, encoding, split y escalado |
| `03_modelos_clasificacion` | ¿Será un hit? — métricas, umbral, lift de negocio |
| `04_modelos_regresion` | ¿Cuánta popularidad tendrá? — error por rango |
| `05_no_supervisado_clustering` | Segmentación musical con K-Means, PCA y DBSCAN |
| `06_optimizacion_ensambles_generalizacion` | Optimización de todos los modelos, Voting/Stacking, generalización, sesgo y selección del modelo final |

> Requiere abrir el notebook con el kernel **Kedro (evaluacion2_machine)** (`kedro jupyter lab`),
> que deja disponibles `session`, `catalog` y `context`.

In [ ]:
%load_ext kedro.ipython

## 1. Elegir qué ejecutar

`PIPELINE = "__default__"` ejecuta todo. Para repetir solo una parte, usar uno de los pipelines registrados
(por ejemplo `"data_preprocessing"`, `"supervised"` o `"unsupervised"`).

In [ ]:
from evaluacion2_machine.pipeline_registry import register_pipelines

pipelines = register_pipelines()
{name: len(p.nodes) for name, p in pipelines.items()}

In [ ]:
PIPELINE = "__default__"
print(f"Se ejecutarán {len(pipelines[PIPELINE].nodes)} nodos del pipeline '{PIPELINE}'")

## 2. Ejecutar

In [ ]:
import time
import warnings

warnings.filterwarnings("ignore")
%reload_kedro
inicio = time.time()
session.run(pipeline_name=PIPELINE)
print(f"Pipeline '{PIPELINE}' terminado en {(time.time() - inicio) / 60:.1f} minutos")

## 3. Verificar los resultados generados

In [ ]:
from pathlib import Path
import pandas as pd

root = Path.cwd().parent
archivos = sorted(p for carpeta in ["models/trained_models", "results/metrics", "results/plots"]
                  for p in (root / carpeta).glob("*") if p.name != ".gitkeep")
pd.DataFrame({
    "archivo": [str(p.relative_to(root)).replace("\\", "/") for p in archivos],
    "KB": [round(p.stat().st_size / 1024, 1) for p in archivos],
    "modificado": [pd.Timestamp(p.stat().st_mtime, unit="s").strftime("%Y-%m-%d %H:%M") for p in archivos],
})

In [ ]:
catalog.load("clf_test_metrics")

In [ ]:
catalog.load("reg_test_metrics")